In [14]:
!pip install pyspark

In [15]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ASS") \
    .getOrCreate()

In [16]:
from pyspark.sql.functions import col, sum, when , trim, lower,  avg, min, max


## read historical file

In [17]:
df = spark.read.csv("historical_data.csv", header=True, inferSchema=True)
df.show()

+-------+------------------+-------------------+-----------------------+-------------------+--------------------+--------------------------+---------------------------+--------------------+--------------------+-----------------+-------------+---------------+-------------------+--------+--------------+-------------------+-------------------+-------------------+
|     ID|Delivery_person_ID|Delivery_person_Age|Delivery_person_Ratings|Restaurant_latitude|Restaurant_longitude|Delivery_location_latitude|Delivery_location_longitude|   Weatherconditions|Road_traffic_density|Vehicle_condition|Type_of_order|Type_of_vehicle|multiple_deliveries|Festival|          City|     Order_DateTime|    Picked_Datetime| Delivered_Datetime|
+-------+------------------+-------------------+-----------------------+-------------------+--------------------+--------------------------+---------------------------+--------------------+--------------------+-----------------+-------------+---------------+----------------

## show schema (data type checking)

In [18]:
df.printSchema()
print(df.columns)

root
 |-- ID: string (nullable = true)
 |-- Delivery_person_ID: string (nullable = true)
 |-- Delivery_person_Age: double (nullable = true)
 |-- Delivery_person_Ratings: double (nullable = true)
 |-- Restaurant_latitude: double (nullable = true)
 |-- Restaurant_longitude: double (nullable = true)
 |-- Delivery_location_latitude: double (nullable = true)
 |-- Delivery_location_longitude: double (nullable = true)
 |-- Weatherconditions: string (nullable = true)
 |-- Road_traffic_density: string (nullable = true)
 |-- Vehicle_condition: integer (nullable = true)
 |-- Type_of_order: string (nullable = true)
 |-- Type_of_vehicle: string (nullable = true)
 |-- multiple_deliveries: double (nullable = true)
 |-- Festival: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Order_DateTime: timestamp (nullable = true)
 |-- Picked_Datetime: timestamp (nullable = true)
 |-- Delivered_Datetime: timestamp (nullable = true)

['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_

## show unique value for categorical data 

In [19]:
df_show = ["Weatherconditions",
           "Road_traffic_density",
           "Vehicle_condition",
           "Type_of_order",
           "Type_of_vehicle",
           "multiple_deliveries",
           "Festival",
           "City"]
for c in df_show:
    print(f"Unique values in column: {c}")
    df.select(c).distinct().show(truncate=False)

Unique values in column: Weatherconditions
+---------------------+
|Weatherconditions    |
+---------------------+
|conditions Sandstorms|
|conditions Fog       |
|conditions Windy     |
|conditions Sunny     |
|conditions NaN       |
|conditions Cloudy    |
|conditions Stormy    |
+---------------------+

Unique values in column: Road_traffic_density
+--------------------+
|Road_traffic_density|
+--------------------+
|Medium              |
|NaN                 |
|Low                 |
|High                |
|Jam                 |
+--------------------+

Unique values in column: Vehicle_condition
+-----------------+
|Vehicle_condition|
+-----------------+
|1                |
|3                |
|2                |
|0                |
+-----------------+

Unique values in column: Type_of_order
+-------------+
|Type_of_order|
+-------------+
|Snack        |
|Buffet       |
|Drinks       |
|Meal         |
+-------------+

Unique values in column: Type_of_vehicle
+-----------------+
|Type

## standardize categorical data (nominal categorical and ordinal categorical, not include ordinal numeric)

In [20]:

categorical_cols = [
    "Weatherconditions",
    "Road_traffic_density",
    "Type_of_order",
    "Type_of_vehicle",
    "Festival",
    "City"
]

for c in categorical_cols:
    df = df.withColumn(c, trim(col(c)))   #remove space
    df = df.withColumn(c, lower(col(c)))

## remove duplicate

In [21]:
# count before removing duplicates
before_count = df.count()
print (before_count)

# remove duplicate
df = df.dropDuplicates()

# count after removing duplicates
after_count = df.count()

removed_duplicated_count = before_count - after_count

print("Rows before removing duplicates:", before_count)
print("Rows after removing duplicates:", after_count)
print("Duplicate rows removed:", removed_duplicated_count)



17097
Rows before removing duplicates: 17097
Rows after removing duplicates: 17097
Duplicate rows removed: 0


## show missing data

In [22]:

all_data = ["ID",
            "Delivery_person_ID",
            "Delivery_person_Age",
            "Delivery_person_Ratings",
            "Restaurant_latitude","Restaurant_longitude",
            "Delivery_location_latitude","Delivery_location_longitude",
            "Weatherconditions",
            "Road_traffic_density",
            "Vehicle_condition",
            "Type_of_order",
            "Type_of_vehicle",
            "multiple_deliveries",
            "Festival",
            "City",
            "Order_DateTime",
            "Picked_Datetime","Delivered_Datetime"]

for c in all_data:
    missing_data =  df.filter(
                        col(c).isNull() |
                        (trim(col(c).cast("string")) == "") |
                        (lower(trim(col(c).cast("string"))).isin("null", "na", "n/a", "nan"))
                    ).count()
    print("Column:",c)
    print("total number of missing data:",missing_data)
    print()

Column: ID
total number of missing data: 0

Column: Delivery_person_ID
total number of missing data: 0

Column: Delivery_person_Age
total number of missing data: 657

Column: Delivery_person_Ratings
total number of missing data: 727

Column: Restaurant_latitude
total number of missing data: 0

Column: Restaurant_longitude
total number of missing data: 0

Column: Delivery_location_latitude
total number of missing data: 0

Column: Delivery_location_longitude
total number of missing data: 0

Column: Weatherconditions
total number of missing data: 0

Column: Road_traffic_density
total number of missing data: 196

Column: Vehicle_condition
total number of missing data: 0

Column: Type_of_order
total number of missing data: 0

Column: Type_of_vehicle
total number of missing data: 0

Column: multiple_deliveries
total number of missing data: 364

Column: Festival
total number of missing data: 78

Column: City
total number of missing data: 426

Column: Order_DateTime
total number of missing dat

## remove row data for important variable that missing

In [23]:
# direct remove for important variable
df_bfr_remove = df.count()

critical_cols = [
    "Delivery_person_ID",
    "Order_DateTime",
    "Picked_Datetime",
    "Delivered_Datetime",
    "Delivery_person_Ratings"
]

for c in critical_cols:
    df = df.withColumn(
        c,
        when(
            col(c).isNull() |
            (trim(col(c)) == "") |
            (lower(trim(col(c))).isin("null", "na", "n/a", "nan")),
            None
        ).otherwise(col(c))
    )

df = df.na.drop(subset=critical_cols)
df_after_remove = df.count()
print("Rows before removing missing value:", df_bfr_remove)
print("Rows after removing missing value:", df_after_remove)

#Delivery_person_Age ,Vehicle_condition,multiple_deliveries

# since Restaurant_latitude, Restaurant_longitude, Delivery_location_latitude, Delivery_location_longitude no used 
# so just leave it when got missing value to prevet too many information dissapear 

#Picked_Datetime,Delivered_Datetime

Rows before removing missing value: 17097
Rows after removing missing value: 16336


## replace all missing categorical data with "unknown"

In [24]:

non_critical_category=["Weatherconditions","Road_traffic_density","Type_of_order","Type_of_vehicle","Festival","City"]


for c in non_critical_category:
    
    count_bfr = df.filter(
        col(c).isNull() |
        (trim(col(c)) == "") |
        (lower(trim(col(c))).isin("null", "na", "n/a", "nan"))
    ).count()
    
    df = df.withColumn(
        c,
        when(
            col(c).isNull() |
            (trim(col(c)) == "") |
            (lower(trim(col(c))).isin("null", "na", "n/a", "nan")),
            "Unknown"
        ).otherwise(col(c))
    )

    count_after = df.filter(
        col(c).isNull() |
        (trim(col(c)) == "") |
        (lower(trim(col(c))).isin("null", "na", "n/a", "nan"))
    ).count()

    print (c)
    print ("Before count for missing data :",count_bfr)
    print ("After count for missing data :",count_after)
print("row count:", df.count())

Weatherconditions
Before count for missing data : 0
After count for missing data : 0
Road_traffic_density
Before count for missing data : 0
After count for missing data : 0
Type_of_order
Before count for missing data : 0
After count for missing data : 0
Type_of_vehicle
Before count for missing data : 0
After count for missing data : 0
Festival
Before count for missing data : 73
After count for missing data : 0
City
Before count for missing data : 399
After count for missing data : 0
row count: 16336


## replace all missing ordinary data with median 

In [25]:
# since cannot replace by unknown so just replace by median

ordinary_data = ["multiple_deliveries","Vehicle_condition"]

for c in ordinary_data:
    
    missing_data_bfr =  df.filter(
        col(c).isNull() |
        (trim(col(c).cast("string")) == "") |
        (lower(trim(col(c).cast("string"))).isin("null", "na", "n/a", "nan"))
    ).count()


    df = df.withColumn(
        c,
        when(
            col(c).isNull() |
            (trim(col(c).cast("string")) == "") |
            (lower(trim(col(c).cast("string"))).isin("null", "na", "n/a", "nan")),
            None
        ).otherwise(col(c))
    
    )
    q1, median_val, q3 = df.filter(col(c).isNotNull()).approxQuantile(c, [0.25, 0.5, 0.75], 0.01)

    df = df.withColumn(
        c,
        when(
            col(c).isNull() |
            (trim(col(c).cast("string")) == "") |
            (lower(trim(col(c).cast("string"))).isin("null", "na", "n/a", "nan")),
            median_val
        ).otherwise(col(c))
    )
    

    missing_data_after = df.filter(
        col(c).isNull() |
        (trim(col(c).cast("string")) == "") |
        (lower(trim(col(c).cast("string"))).isin("null", "na", "n/a", "nan"))
    ).count()
    

    print ("Coloumn: ", c)
    print ("missing data before replace: ", missing_data_bfr)
    print ("missing data after replace", missing_data_after)
    
print("row count:", df.count())


Coloumn:  multiple_deliveries
missing data before replace:  340
missing data after replace 0
Coloumn:  Vehicle_condition
missing data before replace:  0
missing data after replace 0
row count: 16336


## replace all numerical data with mean

In [26]:


mean_val = df.select(avg("Delivery_person_Age")).collect()[0][0]
q1, median_val, q3 = df.approxQuantile("Delivery_person_Age", [0.25, 0.5, 0.75], 0.01)
min_val = df.select(min("Delivery_person_Age")).collect()[0][0]
max_val = df.select(max("Delivery_person_Age")).collect()[0][0]

print("Delivery_person_Age")
print("Min:", min_val)
print("Q1:", q1)
print("Median:", median_val)
print("Q3:", q3)
print("Mean:", mean_val)
print("Max:", max_val)

# since median is close to mean, use mean to handle missing values

# count missing before
missing_before = df.filter(
    col("Delivery_person_Age").isNull() |
    (trim(col("Delivery_person_Age").cast("string")) == "") |
    (lower(trim(col("Delivery_person_Age").cast("string"))).isin("null", "na", "n/a", "nan"))
).count()

# replace missing with mean
df = df.withColumn(
    "Delivery_person_Age",
    when(
        col("Delivery_person_Age").isNull() |
        (trim(col("Delivery_person_Age").cast("string")) == "") |
        (lower(trim(col("Delivery_person_Age").cast("string"))).isin("null", "na", "n/a", "nan")),
        mean_val
    ).otherwise(col("Delivery_person_Age"))
)

# count missing after
missing_after = df.filter(
    col("Delivery_person_Age").isNull() |
    (trim(col("Delivery_person_Age").cast("string")) == "") |
    (lower(trim(col("Delivery_person_Age").cast("string"))).isin("null", "na", "n/a", "nan"))
).count()

print()
print("Mean used for replace:", mean_val)
print("Missing before:", missing_before)
print("Missing after:", missing_after)

print("row count:", df.count())

Delivery_person_Age
Min: 20.0
Q1: 25.0
Median: 29.0
Q3: 34.0
Mean: 29.575538687561213
Max: 39.0

Mean used for replace: 29.575538687561213
Missing before: 0
Missing after: 0
row count: 16336


## check for non-critical data (scare remove too many data if directly remove all invalid )  now handle by remove all first not sure

In [27]:
non_critical_data = ["Restaurant_latitude","Restaurant_longitude",
                     "Delivery_location_latitude","Delivery_location_longitude"]
    
for c in non_critical_data:
    
    missing_data_count_bfr = df.filter(
        col(c).isNull() |
        (trim(col(c).cast("string")) == "") |
        (lower(trim(col(c).cast("string"))).isin("null", "na", "n/a", "nan"))
    ).count()
    
    df = df.withColumn(
        c,
        when(
            col(c).isNull() |
            (trim(col(c)) == "") |
            (lower(trim(col(c))).isin("null", "na", "n/a", "nan")),
            None
        ).otherwise(col(c))
    )

    missing_data_count_after = df.filter(
        col(c).isNull() |
        (trim(col(c).cast("string")) == "") |
        (lower(trim(col(c).cast("string"))).isin("null", "na", "n/a", "nan"))
    ).count()

    print(c)
    print("total number of missing data before cleaning:", missing_data_count_bfr)
    print("total number of missing data after cleaning:", missing_data_count_after)
    print()

df = df.na.drop(subset=critical_cols)

print("row count:", df.count())

Restaurant_latitude
total number of missing data before cleaning: 0
total number of missing data after cleaning: 0

Restaurant_longitude
total number of missing data before cleaning: 0
total number of missing data after cleaning: 0

Delivery_location_latitude
total number of missing data before cleaning: 0
total number of missing data after cleaning: 0

Delivery_location_longitude
total number of missing data before cleaning: 0
total number of missing data after cleaning: 0

row count: 16336


# range check

Missing values were imputed because the remaining fields in the record could still be valid and useful. However, rows containing invalid values were removed, since such values indicate data quality errors and reduce confidence in the reliability of the entire record. values outside predefined valid ranges are treated as invalid and the affected records are filtered out.

In [29]:
vehicle_out_of_range_couunt_bfr = df.filter((col("Vehicle_condition") < 0) | (col("Vehicle_condition") > 3)).count()
rating_out_of_range_count_bfr = df.filter((col("Delivery_person_Ratings") < 1) | (col("Delivery_person_Ratings") > 5)).count()
age_out_of_range_count_bfr = df.filter((col("Delivery_person_Age") < 18) | (col("Delivery_person_Age") > 65)).count()
multiple_deliveries_out_of_range_count_bfr = df.filter(col("multiple_deliveries") < 0).count()

# remove rows that contain any out-of-range value
df = df.filter(
    (col("Vehicle_condition") >= 0) & (col("Vehicle_condition") <= 3) &
    (col("Delivery_person_Ratings") >= 1) & (col("Delivery_person_Ratings") <= 5) &
    (col("Delivery_person_Age") >= 18) & (col("Delivery_person_Age") <= 65) &
    (col("multiple_deliveries") >= 0)
)

vehicle_out_of_range_couunt_after = df.filter((col("Vehicle_condition") < 0) | (col("Vehicle_condition") > 3)).count()
rating_out_of_range_count_after = df.filter((col("Delivery_person_Ratings") < 1) | (col("Delivery_person_Ratings") > 5)).count()
age_out_of_range_count_after = df.filter((col("Delivery_person_Age") < 18) | (col("Delivery_person_Age") > 65)).count()
multiple_deliveries_out_of_range_count_after = df.filter(col("multiple_deliveries") < 0).count()


print ("number vehicle condition that out of range before : ",vehicle_out_of_range_couunt_bfr)
print ("number vehicle condition that out of range after: ",vehicle_out_of_range_couunt_after)
print()
print ("number delivery person ratings that out of range before : ",rating_out_of_range_count_bfr)
print ("number delivery person ratings that out of range after : ",rating_out_of_range_count_after)
print()
print ("number delivery person age that out of range before : ",age_out_of_range_count_bfr)
print ("number delivery person age that out of range after : ",age_out_of_range_count_after)
print()
print ("number multiple deliveries that out of range before : ",multiple_deliveries_out_of_range_count_bfr)
print ("number multiple deliveries that out of range after : ",multiple_deliveries_out_of_range_count_after)

print("row count:", df.count())

number vehicle condition that out of range before :  0
number vehicle condition that out of range after:  0

number delivery person ratings that out of range before :  0
number delivery person ratings that out of range after :  0

number delivery person age that out of range before :  0
number delivery person age that out of range after :  0

number multiple deliveries that out of range before :  0
number multiple deliveries that out of range after :  0
row count: 16336


## timestamp check

In [30]:
invalid_timestamp_order_before = df.filter(
    (col("Picked_Datetime") < col("Order_DateTime")) |
    (col("Delivered_Datetime") < col("Picked_Datetime"))
).count()

invalid_timestamp_order_before_list = df.filter(
    (col("Picked_Datetime") < col("Order_DateTime")) |
    (col("Delivered_Datetime") < col("Picked_Datetime"))
)


df = df.filter(
    (col("Picked_Datetime") >= col("Order_DateTime")) &
    (col("Delivered_Datetime") >= col("Picked_Datetime"))
)

invalid_timestamp_order_after = df.filter(
    (col("Picked_Datetime") < col("Order_DateTime")) |
    (col("Delivered_Datetime") < col("Picked_Datetime"))
).count()


# show some invalid examples
print("Example invalid timestamp rows:")
invalid_timestamp_order_before_list.select(
    "Order_DateTime",
    "Picked_Datetime",
    "Delivered_Datetime"
).show(10, False)
print("invalid timestamp order before remove:", invalid_timestamp_order_before)
print("invalid timestamp order after remove:", invalid_timestamp_order_after)
print("row count:", df.count())

Example invalid timestamp rows:
+-------------------+-------------------+-------------------+
|Order_DateTime     |Picked_Datetime    |Delivered_Datetime |
+-------------------+-------------------+-------------------+
|2022-02-12 23:50:00|2022-02-12 00:00:00|2022-02-12 00:12:00|
|2022-02-18 23:45:00|2022-02-18 00:00:00|2022-02-18 00:28:00|
|2022-03-01 23:55:00|2022-03-01 00:10:00|2022-03-01 00:22:00|
|2022-03-03 23:55:00|2022-03-03 00:10:00|2022-03-03 00:39:00|
|2022-03-05 23:55:00|2022-03-05 00:10:00|2022-03-05 00:55:00|
|2022-02-12 23:50:00|2022-02-12 00:05:00|2022-02-12 00:50:00|
|2022-02-16 23:55:00|2022-02-16 00:05:00|2022-02-16 00:15:00|
|2022-02-18 23:55:00|2022-02-18 00:00:00|2022-02-18 00:32:00|
|2022-03-03 23:55:00|2022-03-03 00:05:00|2022-03-03 00:39:00|
|2022-02-14 23:50:00|2022-02-14 00:00:00|2022-02-14 00:31:00|
+-------------------+-------------------+-------------------+
only showing top 10 rows
invalid timestamp order before remove: 325
invalid timestamp order after re

## coordinate check

In [31]:
# before remove
restaurant_lat_out_of_range_before = df.filter(
    (col("Restaurant_latitude") < -90) | (col("Restaurant_latitude") > 90)
).count()

restaurant_long_out_of_range_before = df.filter(
    (col("Restaurant_longitude") < -180) | (col("Restaurant_longitude") > 180)
).count()

delivery_lat_out_of_range_before = df.filter(
    (col("Delivery_location_latitude") < -90) | (col("Delivery_location_latitude") > 90)
).count()

delivery_long_out_of_range_before = df.filter(
    (col("Delivery_location_longitude") < -180) | (col("Delivery_location_longitude") > 180)
).count()

# remove rows with invalid coordinates
df = df.filter(
    (col("Restaurant_latitude") >= -90) & (col("Restaurant_latitude") <= 90) &
    (col("Restaurant_longitude") >= -180) & (col("Restaurant_longitude") <= 180) &
    (col("Delivery_location_latitude") >= -90) & (col("Delivery_location_latitude") <= 90) &
    (col("Delivery_location_longitude") >= -180) & (col("Delivery_location_longitude") <= 180)
)

# after remove
restaurant_lat_out_of_range_after = df.filter(
    (col("Restaurant_latitude") < -90) | (col("Restaurant_latitude") > 90)
).count()

restaurant_long_out_of_range_after = df.filter(
    (col("Restaurant_longitude") < -180) | (col("Restaurant_longitude") > 180)
).count()

delivery_lat_out_of_range_after = df.filter(
    (col("Delivery_location_latitude") < -90) | (col("Delivery_location_latitude") > 90)
).count()

delivery_long_out_of_range_after = df.filter(
    (col("Delivery_location_longitude") < -180) | (col("Delivery_location_longitude") > 180)
).count()

print("Before removal")
print("restaurant latitude out of range:", restaurant_lat_out_of_range_before)
print("restaurant longitude out of range:", restaurant_long_out_of_range_before)
print("delivery latitude out of range:", delivery_lat_out_of_range_before)
print("delivery longitude out of range:", delivery_long_out_of_range_before)
print()

print("After removal")
print("restaurant latitude out of range:", restaurant_lat_out_of_range_after)
print("restaurant longitude out of range:", restaurant_long_out_of_range_after)
print("delivery latitude out of range:", delivery_lat_out_of_range_after)
print("delivery longitude out of range:", delivery_long_out_of_range_after)

Before removal
restaurant latitude out of range: 0
restaurant longitude out of range: 0
delivery latitude out of range: 0
delivery longitude out of range: 0

After removal
restaurant latitude out of range: 0
restaurant longitude out of range: 0
delivery latitude out of range: 0
delivery longitude out of range: 0


## create new variable

In [32]:
from pyspark.sql.functions import col, hour, to_date, date_format

df = df.withColumn(
    "pickup_delay_minutes",
    (col("Picked_Datetime").cast("long") - col("Order_DateTime").cast("long")) / 60
).withColumn(
    "delivery_minutes",
    (col("Delivered_Datetime").cast("long") - col("Picked_Datetime").cast("long")) / 60
).withColumn(
    "order_date",
    to_date(col("Order_DateTime"))
)

df.select(
    "Order_DateTime",
    "Picked_Datetime",
    "Delivered_Datetime",
    "pickup_delay_minutes",
    "delivery_minutes",
    "order_date"
).show(10, truncate=False)

print("row count:", df.count())

+-------------------+-------------------+-------------------+--------------------+----------------+----------+
|Order_DateTime     |Picked_Datetime    |Delivered_Datetime |pickup_delay_minutes|delivery_minutes|order_date|
+-------------------+-------------------+-------------------+--------------------+----------------+----------+
|2022-02-11 09:25:00|2022-02-11 09:35:00|2022-02-11 09:45:00|10.0                |10.0            |2022-02-11|
|2022-02-11 09:30:00|2022-02-11 09:45:00|2022-02-11 10:18:00|15.0                |33.0            |2022-02-11|
|2022-02-11 09:55:00|2022-02-11 10:00:00|2022-02-11 10:23:00|5.0                 |23.0            |2022-02-11|
|2022-02-11 10:55:00|2022-02-11 11:10:00|2022-02-11 11:44:00|15.0                |34.0            |2022-02-11|
|2022-02-11 10:55:00|2022-02-11 11:10:00|2022-02-11 11:28:00|15.0                |18.0            |2022-02-11|
|2022-02-11 11:15:00|2022-02-11 11:20:00|2022-02-11 12:05:00|5.0                 |45.0            |2022-02-11|
|

## summary

In [33]:
from pyspark.sql.functions import avg, countDistinct, expr

kpi_summary = df.agg(
    expr("count(*)").alias("total_orders"),
    avg("delivery_minutes").alias("avg_delivery_minutes"),
    avg("pickup_delay_minutes").alias("avg_pickup_delay_minutes"),
    avg("Delivery_person_Ratings").alias("avg_rider_rating"),
    countDistinct("Delivery_person_ID").alias("distinct_delivery_persons")
)

kpi_summary.show(truncate=False)

+------------+--------------------+------------------------+-----------------+-------------------------+
|total_orders|avg_delivery_minutes|avg_pickup_delay_minutes|avg_rider_rating |distinct_delivery_persons|
+------------+--------------------+------------------------+-----------------+-------------------------+
|16011       |27.46742864280807   |9.973768034476297       |4.638873274623696|1320                     |
+------------+--------------------+------------------------+-----------------+-------------------------+



## group by city

In [34]:
from pyspark.sql.functions import avg, count

city_analysis = df.groupBy("City").agg(
    count("*").alias("total_orders"),
    avg("delivery_minutes").alias("avg_delivery_minutes"),
    avg("pickup_delay_minutes").alias("avg_pickup_delay_minutes"),
    avg("Delivery_person_Ratings").alias("avg_rider_rating")
).orderBy("avg_delivery_minutes", ascending=False)

city_analysis.show(truncate=False)

+-------------+------------+--------------------+------------------------+-----------------+
|City         |total_orders|avg_delivery_minutes|avg_pickup_delay_minutes|avg_rider_rating |
+-------------+------------+--------------------+------------------------+-----------------+
|urban        |3544        |27.70372460496614   |10.090293453724605      |4.682251693002252|
|metropolitian|12020       |27.414559068219635  |9.940931780366057       |4.625299500831951|
|Unknown      |391         |27.0383631713555    |9.948849104859335       |4.68312020460358 |
|semi-urban   |56          |26.857142857142858  |9.821428571428571       |4.498214285714285|
+-------------+------------+--------------------+------------------------+-----------------+



## group by weather

In [35]:
weather_analysis = df.groupBy("Weatherconditions").agg(
    count("*").alias("total_orders"),
    avg("delivery_minutes").alias("avg_delivery_minutes"),
    avg("pickup_delay_minutes").alias("avg_pickup_delay_minutes"),
    avg("Delivery_person_Ratings").alias("avg_rider_rating")
).orderBy("avg_delivery_minutes", ascending=False)

weather_analysis.show(truncate=False)

+---------------------+------------+--------------------+------------------------+-----------------+
|Weatherconditions    |total_orders|avg_delivery_minutes|avg_pickup_delay_minutes|avg_rider_rating |
+---------------------+------------+--------------------+------------------------+-----------------+
|conditions windy     |2690        |27.746840148698883  |9.934944237918216       |4.629776951672852|
|conditions cloudy    |2646        |27.668556311413454  |10.049130763416478      |4.651625094482239|
|conditions sandstorms|2639        |27.619931792345586  |9.907161803713528       |4.621636983705945|
|conditions fog       |2630        |27.36425855513308   |10.013307984790874      |4.658403041825097|
|conditions stormy    |2743        |27.23076923076923   |10.016405395552315      |4.616842872767043|
|conditions sunny     |2663        |27.179872324446112  |9.92114156965828        |4.655876830642116|
+---------------------+------------+--------------------+------------------------+---------

## group by vehicle type

In [ ]:
vehicle_analysis = df.groupBy("Type_of_vehicle").agg(
    count("*").alias("total_orders"),
    avg("delivery_minutes").alias("avg_delivery_minutes"),
    avg("pickup_delay_minutes").alias("avg_pickup_delay_minutes"),
    avg("Delivery_person_Ratings").alias("avg_rider_rating")
).orderBy("avg_delivery_minutes", ascending=False)

vehicle_analysis.show(truncate=False)

## group by type of order

In [ ]:
traffic_analysis = df.groupBy("Type_of_order").agg(
    count("*").alias("total_orders"),
    avg("delivery_minutes").alias("avg_delivery_minutes"),
    avg("pickup_delay_minutes").alias("avg_pickup_delay_minutes"),
    avg("Delivery_person_Ratings").alias("avg_rider_rating")
).orderBy("avg_delivery_minutes", ascending=False)

traffic_analysis.show(truncate=False)

## save to csv (summary and all group info)